In [19]:
NUM_AGENTS = 7
HEIGHT = 12
WIDTH = 12
SPAWN_PROB_PER_CELL = 0.05
DESPAWN_PROB_PER_CELL = 0.09
# Parameters
ENV_ITERATIONS = 100000
DISCOUNT = 0.99
EPSILON = 0.1
LEARNING_RATE = 0.00002
BATCH_SIZE = 4

MLP_INPUT_DIM = 2 * HEIGHT * WIDTH
HIDDEN_FEATURES = 128
HIDDEN_LAYERS = 4
FILENAME = "results_centralized.csv"

NUM_ENV_CHECKPOINTS = 10

In [20]:
import sys
sys.path.append("../..")
import numpy as np
from orchard.environment import Action2D, OrchardBasic
from tqdm import tqdm
import torch
from scipy.stats import norm
import copy
import numpy as np
import torch
import matplotlib.pyplot as plt

from models.value_cnn import Transition
from utils import ten_float
from dataclasses import dataclass
from typing import Any
from env_functions import *
from eval_functions import *
import csv
from pathlib import Path
from models.value_function import VNetwork
from helpers.controllers import ViewController 


In [21]:
viewController = ViewController(vision=0, new_input=False)

def raw_state_for_mlp(state_dict):
    # This function replicates the input processing used in the MLP experiments
    return viewController.state_to_nn_input(state_dict, None, None).flatten()

def state_to_raw_dict(s: State) -> dict:
    return {"apples": s.apples, "agents": s.agents}

In [22]:
def simulate_step(s: State, agent_idx: int, agent_positions: np.ndarray, action_vector: np.ndarray):
    """
    Simulates an agent taking an action. Does not modify in place.
    
    Returns:
        tuple: (reward: int, next_state: State, new_agent_positions: np.ndarray)
    """
    current_agents = s.agents
    current_apples = s.apples
    agent_pos = agent_positions[agent_idx]
    grid_shape = current_agents.shape
    
    new_position = np.clip(
        agent_pos + action_vector, [0, 0], np.array(grid_shape) - 1
    )
    
    next_agents = current_agents.copy()
    next_apples = current_apples.copy()
    
    next_agents[tuple(new_position)] += 1
    next_agents[tuple(agent_pos)] -= 1
    
    # The new positions array must be updated
    new_agent_positions = agent_positions.copy()
    new_agent_positions[agent_idx] = new_position
    
    reward = 0
    if next_apples[tuple(new_position)] > 0:
        next_apples[tuple(new_position)] -= 1
        reward = 1
        
    return reward, State(apples=next_apples, agents=next_agents), new_agent_positions

### Get CNN Centralized Estimate Value

In [23]:
def Q_team(s_t_plus_1: State, r_team: float, network: VNetwork):
    return r_team + DISCOUNT * network.get_value_function(raw_state_for_mlp(state_to_raw_dict(s_t_plus_1)))

def argmax_a_of_Q_team(s_t, agent_idx: int, agent_positions: np.ndarray, network: VNetwork):
    max_q = -float('inf')
    best_action = Action2D.STAY
    for action in Action2D:
        r_t, s_t_plus_1, new_agent_positions = simulate_step(s_t, agent_idx, agent_positions, action.vector)
        q_value = Q_team(s_t_plus_1, r_t, network)
        if q_value > max_q:
            max_q = q_value
            best_action = action
    return best_action

In [24]:
# Instantiate the VNetwork (MLP) model
value_network = VNetwork(MLP_INPUT_DIM, 1, LEARNING_RATE, 
                         DISCOUNT, HIDDEN_FEATURES, HIDDEN_LAYERS)

s_0 = init_empty_state(HEIGHT, WIDTH)
place_agents_randomly(s_0, NUM_AGENTS)
agent_positions = np.argwhere(s_0.agents == 1)

TARGET_UPDATE_FREQUENCY = 100

total_steps = 0

s_t = s_0.copy()
for t in tqdm(range(ENV_ITERATIONS), desc="Training"):
    c = np.random.randint(0, len(agent_positions))
    
    p = np.random.random()
    if p < EPSILON:
        action = Action2D.get_random_action()
    else:
        action = argmax_a_of_Q_team(s_t, c, agent_positions, value_network)
        
    r_t, s_intermediate, agent_positions = simulate_step(s_t, c, agent_positions, action.vector)
    s_t_plus_1 = s_intermediate.copy()
    spawn_apples(s_t_plus_1, SPAWN_PROB_PER_CELL)
    despawn_apples(s_t_plus_1, DESPAWN_PROB_PER_CELL)

    # 1. Convert states to the network input format
    processed_s_t = raw_state_for_mlp(state_to_raw_dict(s_t))
    processed_s_t_plus_1 = raw_state_for_mlp(state_to_raw_dict(s_t_plus_1))

    # 2. Add the experience to the replay buffer
    value_network.add_experience(processed_s_t, processed_s_t_plus_1, r_t)
    if len(value_network.batch_states) >= BATCH_SIZE:
        value_network.train()


    s_t = s_t_plus_1
    total_steps += 1




Training:   0%|          | 0/100000 [00:00<?, ?it/s]

Training:  11%|█▏        | 11255/100000 [00:58<07:44, 191.22it/s]


KeyboardInterrupt: 

In [ ]:
def optimal_learned_policy(s: State, agent_idx: int, agent_positions: np.ndarray):
    return argmax_a_of_Q_team(s, agent_idx, agent_positions, value_network)

In [ ]:
def evaluate_policy(policy_func, num_steps: int = 1000):
    total_rewards = []
    s_0 = init_empty_state(HEIGHT, WIDTH)
    place_agents_randomly(s_0, NUM_AGENTS)
    agent_positions = np.argwhere(s_0.agents == 1)
    s_t = s_0.copy()
    for t in range(num_steps):  # fixed episode length
        c = np.random.randint(0, len(agent_positions))
        action = policy_func(s_t, c, agent_positions)
        r_t, s_intermediate, agent_positions = simulate_step(s_t, c, agent_positions, action.vector)
        s_t_plus_1 = s_intermediate.copy()
        spawn_apples(s_t_plus_1, SPAWN_PROB_PER_CELL)
        despawn_apples(s_t_plus_1, DESPAWN_PROB_PER_CELL)
        total_rewards.append(r_t)
        s_t = s_t_plus_1
    return total_rewards

In [ ]:
random_rewards = evaluate_policy(random_policy, num_steps=1000)
nearest_rewards = evaluate_policy(nearest_policy, num_steps=1000)
learned_rewards = evaluate_policy(optimal_learned_policy, num_steps=1000)
print(f"Random Policy: Mean Reward = {np.mean(random_rewards):.4f}, Std Dev = {np.std(random_rewards):.4f}")
print(f"Nearest Policy: Mean Reward = {np.mean(nearest_rewards):.4f}, Std Dev = {np.std(nearest_rewards):.4f}")
print(f"Learned Policy: Mean Reward = {np.mean(learned_rewards):.4f}, Std Dev = {np.std(learned_rewards):.4f}")

Random Policy: Mean Reward = 0.1870, Std Dev = 0.3899
Nearest Policy: Mean Reward = 0.6230, Std Dev = 0.4846
Learned Policy: Mean Reward = 0.6520, Std Dev = 0.4763
